<a href="https://colab.research.google.com/github/sameds1011/File-Based-Content-Analysis-and-Summarization-with-NLP/blob/main/otomasyonyeni_ipynb_adl_not_defterinin_kopyas_adl_not_defterinin_kopyas_adl_not_defterinin_kopyas.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
pip install fpdf python-docx PyPDF2 gradio


  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 244.3/244.3 kB 11.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 232.6/232.6 kB 10.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 57.5/57.5 MB 7.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 320.6/320.6 kB 8.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 94.8/94.8 kB 6.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.3/11.3 MB 36.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.2/73.2 kB 4.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 62.3/62.3 kB 3.8 MB/s eta 0:00:00
  Created wheel for fpdf: filename=fpdf-1.7.2-py2.py3-none-any.whl size=40704 sha256=c763d03df4fc98f830389413d13497b4627d2803f8c194489da961fa1c963ba6
  Stored in directory: /root/.cache/pip/wheels/f9/95/ba/f418094659025eb9611f17cbcaf2334236bf39a0c3453ea455
Successfully built fpdf
  Attempting uninstall: markupsafe
    Found 

In [ ]:
import re
import gradio as gr
from PyPDF2 import PdfReader
from docx import Document
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

# PDF'den Metin Çıkarma Fonksiyonu
def pdf_metin_cikarma(file_path):
    try:
        reader = PdfReader(file_path)
        text = ""
        for page in reader.pages:
            text += page.extract_text()
        return text.strip()
    except Exception as e:
        return f"Error extracting text from PDF: {str(e)}"

# Word'den Metin Çıkarma Fonksiyonu
def word_metin_cikarma(file_path):
    try:
        doc = Document(file_path)
        text = ""
        for paragraph in doc.paragraphs:
            text += paragraph.text + "\n"
        return text.strip()
    except Exception as e:
        return f"Error extracting text from Word document: {str(e)}"

# Cümle Bölme Fonksiyonu (Regex Kullanılarak)
def cumlelere_bol(text):
    try:
        sentences = re.split(r'(?<=[.!?]) +', text)
        return [sentence.strip() for sentence in sentences if sentence.strip()]
    except Exception as e:
        return f"Error splitting sentences: {str(e)}"

# Metin Temizleme Fonksiyonu
def metin_temizleme(text):
    try:
        text = re.sub(r'http\S+', '', text)  # URL'leri temizle
        text = re.sub(r'\s+', ' ', text)  # Fazla boşlukları temizle
        return text.strip()
    except Exception as e:
        return f"Error cleaning text: {str(e)}"

# Konu Başlıklarını Çıkarma ve Gruplandırma Fonksiyonu
def metni_gruplama(text):
    try:
        structured_data = {
            "Topics": [],
            "Introduction": [],
            "Purpose and Goal": [],
            "Conclusion": [],
            "Importance of the Topic": [],
            "Why Important": [],
            "What We Learned": [],
            "Summary": []
        }

        sentences = cumlelere_bol(text)

        # Başlıkların sınıflandırılması
        keywords = {
            "Introduction": ["introduction", "overview", "background", "context"],
            "Purpose and Goal": ["purpose", "goal", "aim", "objective"],
            "Conclusion": ["conclusion", "summary", "closing"],
            "Importance of the Topic": ["important", "significant", "critical"],
            "Why Important": ["why", "reason", "essential"],
            "What We Learned": ["learned", "takeaways", "insights"]
        }

        current_topic = "Topics"
        for line in sentences:
            line_lower = line.lower()
            for topic, topic_keywords in keywords.items():
                if any(keyword in line_lower for keyword in topic_keywords):
                    current_topic = topic
                    break
            structured_data[current_topic].append(line)

        # Kısa cümleleri "Topics" için seçme
        topics_detected = [sentence for sentence in sentences if len(sentence.split()) <= 8 and any(c.isalpha() for c in sentence)]
        structured_data["Topics"] = list(set(topics_detected))

        return structured_data
    except Exception as e:
        return f"Error grouping text: {str(e)}"

# Başlık ve İçerik Uyumu Değerlendirme
def uyum_puani_hesapla(grouped_data):
    try:
        vectorizer = TfidfVectorizer()
        uyum_puanlari = {}

        for topic, content in grouped_data.items():
            if content:
                topic_vector = vectorizer.fit_transform([topic])
                content_vector = vectorizer.transform(content)
                similarities = cosine_similarity(topic_vector, content_vector).flatten()
                uyum_puanlari[topic] = round(similarities.mean() * 100, 2) if similarities.size > 0 else 0.0

        return uyum_puanlari
    except Exception as e:
        return f"Error calculating alignment score: {str(e)}"

# Özet Üretme Fonksiyonu
def generate_summary(text):
    try:
        sentences = cumlelere_bol(text)
        if len(sentences) > 5:
            summary = " ".join(sentences[:2] + sentences[-2:])
        else:
            summary = " ".join(sentences)
        return summary
    except Exception as e:
        return f"Error generating summary: {str(e)}"

# Transkript Üretimi Fonksiyonu
def transkript_olusturma(gruplu_veri):
    try:
        transcript = ""
        for topic, content in gruplu_veri.items():
            if content:
                transcript += f"\n### {topic.upper()} ###\n"
                transcript += '\n'.join([f"- {line}" for line in content if line.strip()])
                transcript += "\n"
        return transcript.strip()
    except Exception as e:
        return f"Error creating transcript: {str(e)}"

# Gradio İşlem Fonksiyonu
def process_and_generate_transcript(file):
    try:
        if file.name.endswith(".pdf"):
            content = pdf_metin_cikarma(file.name)
        elif file.name.endswith(".txt"):
            content = file.read().decode('utf-8')
        elif file.name.endswith(".docx"):
            content = word_metin_cikarma(file.name)
        else:
            return "Unsupported file format. Please upload a PDF, TXT, or DOCX file.", None

        cleaned_text = metin_temizleme(content)
        grouped_data = metni_gruplama(cleaned_text)
        if isinstance(grouped_data, str):
            return grouped_data, None

        summary = generate_summary(cleaned_text)
        grouped_data["Summary"].append(summary)

        alignment_scores = uyum_puani_hesapla(grouped_data)
        transcript = transkript_olusturma(grouped_data)

        output_file = "transcript.txt"
        with open(output_file, "w", encoding="utf-8") as f:
            f.write(transcript)

        return transcript, output_file, alignment_scores
    except Exception as e:
        return f"Error during processing: {str(e)}", None, None

# Geliştirilmiş Gradio Arayüzü
with gr.Blocks() as interface:
    gr.Markdown("## Advanced Text Analysis and Transcript Generator")
    gr.Markdown("Upload a PDF, Word (DOCX), or TXT file to generate a structured transcript based on its content.")

    with gr.Row():
        file_input = gr.File(label="Upload Text File (PDF, DOCX, or TXT)", file_types=[".pdf", ".txt", ".docx"])
        submit_btn = gr.Button("Submit")
        clear_btn = gr.Button("Clear")

    transcript_output = gr.Textbox(label="Generated Transcript")
    download_output = gr.File(label="Download Transcript")
    alignment_scores_output = gr.Textbox(label="Alignment Scores")

    submit_btn.click(
        fn=process_and_generate_transcript,
        inputs=file_input,
        outputs=[transcript_output, download_output, alignment_scores_output]
    )

    clear_btn.click(
        fn=lambda: ("", None, ""),
        inputs=None,
        outputs=[transcript_output, download_output, alignment_scores_output]
    )

interface.launch()


Running Gradio in a Colab notebook requires sharing enabled. Automatically setting `share=True` (you can turn this off by setting `share=False` in `launch()` explicitly).

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://2439f9a76556440ddf.gradio.live

This share link expires in 72 hours. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
